# Julia · Kriging vs GaussianProcesses.jl

**Idea.** JLibKriging against the main Julia GP package: default fits on the shared
datasets.

**Reference.** GaussianProcesses.jl (accuracy and cost).

**Tags.** class: Kriging · kernel: matern5_2 · compared to: GaussianProcesses.jl

Default fits on the shared datasets of `data/` (Branin, Hartmann 3, Borehole):
JLibKriging (`matern5_2`, constant trend, single start and 10 restarts) against
[GaussianProcesses.jl](https://github.com/STOR-i/GaussianProcesses.jl)
(Matérn 5/2 ARD, constant mean, MLE), warn-level comparison of Q².

The module is `JLibKriging` (the packaged binding, [JLibKriging.jl](https://github.com/libKriging/JLibKriging.jl)),
imported under a short name (as GaussianProcesses.jl): both export `predict`,
`optimize!`, ... so every call is qualified.

In [1]:
import JLibKriging as LK
include("tools/julia/gallery.jl")
s = Suite("vs-GaussianProcesses_Kriging_julia")

Suite("vs-GaussianProcesses_Kriging_julia", Vector{String}[], Vector{String}[], 1.793005961351359e9, Dict(("large-n_Kriging-vecchia-nystrom_python", "otl_circuit/n=5000/nystrom(100): Q2") => "Nystrom predictor degrades as n grows at fixed k (theta estimates are fine: the exact predictor at the Nystrom theta has Q2 ~ 1)", ("robustness-invalid-inputs_Kriging_python", "Inf in y is rejected") => "Kriging accepts Inf in y without raising", ("large-n_Kriging-vecchia-nystrom_python", "vecchia(n-1) LL == exact LL (same theta)") => "With m >= n-1 neighbours the Vecchia likelihood should be the exact one; it matches for n=20 but not for n=40 (-111.349 vs -110.834)", ("robustness-invalid-inputs_Kriging_python", "NaN in y is rejected") => "Kriging accepts NaN in y without raising", ("robustness-invalid-inputs_Kriging_python", "WarpKriging with 1 spec for 2 columns is rejected") => "WarpKriging accepts fewer warping specs than columns of X", ("robustness-invalid-inputs_Kriging_python", "predict(std

## Installing GaussianProcesses.jl

Added to the active Julia environment (with Optim.jl, its optimizer). Without
it, the comparison is skipped (warn-level).

In [2]:
import Pkg
have_gp = try
    Pkg.add(["GaussianProcesses", "Optim"])
    @eval import GaussianProcesses as GP
    @eval import Optim
    true
catch e
    println("GaussianProcesses.jl could not be installed: ", sprint(showerror, e))
    false
end

    Updating registry at `~/.julia/registries/General.toml`


   Resolving 

package versions...


   Installed IrrationalConstants ────── v0.2.7
   Installed ScikitLearnBase ────────── v0.5.0
   Installed Rmath ──────────────────── v0.9.0
   Installed ElasticPDMats ──────────── v0.2.4
   Installed DiffRules ──────────────── v1.16.0
   Installed HypergeometricFunctions ── v0.3.31
   Installed Adapt ──────────────────── v4.7.4
   Installed StatsFuns ──────────────── v1.5.3
   Installed PtrArrays ──────────────── v1.4.0
   Installed PositiveFactorizations ─── v0.2.4
   Installed DiffResults ────────────── v1.1.0
   Installed ADTypes ────────────────── v1.24.0
   Installed Accessors ──────────────── v0.1.45
   Installed FastGaussQuadrature ────── v1.3.0


   Installed DataAPI ────────────────── v1.16.0
   Installed PDMats ─────────────────── v0.11.41
   Installed Setfield ───────────────── v1.1.2
   Installed RecipesBase ────────────── v1.3.4
   Installed StaticArrays ───────────── v1.9.24


...

    943.9 ms  ✓ LineSearches


   5622.5 ms  ✓ Distributions


   2531.4 ms  ✓ Optim


   2411.6 ms  ✓ GaussianProcesses
  94 dependencies successfully precompiled in 38 seconds. 52 already precompiled.


true

## Default fits

In [3]:
for name in ("branin", "hartmann3", "borehole")
    Xd = read_data(name, "X"); yd = vec(read_data(name, "y")); Xt = read_data(name, "Xt"); yt = vec(read_data(name, "yt"))
    t = @elapsed k = LK.Kriging(yd, Xd, "matern5_2")
    q_lk = q2(yt, LK.predict(k, Xt; return_stdev=false).mean)
    metric!(s, name, "JLibKriging"; q2=q_lk, fit_s=t)
    t = @elapsed k10 = LK.Kriging(yd, Xd, "matern5_2"; optim="BFGS10")
    q_lk10 = q2(yt, LK.predict(k10, Xt; return_stdev=false).mean)
    metric!(s, name, "JLibKriging BFGS10"; q2=q_lk10, fit_s=t)
    check!(s, "$name: Q2(JLibKriging BFGS10) > 0.9", q_lk10 > 0.9; value=q_lk10, expected="> 0.9")
    if have_gp
        try
            d = size(Xd, 2)
            t = @elapsed begin
                gp = GP.GP(permutedims(Xd), yd, GP.MeanConst(sum(yd) / length(yd)),
                           GP.Mat52Ard(zeros(d), log(sum((yd .- sum(yd) / length(yd)) .^ 2) / length(yd)) / 2), -8.0)
                GP.optimize!(gp; noise=false)
            end
            mg, _ = GP.predict_f(gp, permutedims(Xt))
            q_gp = q2(yt, mg)
            metric!(s, name, "GaussianProcesses.jl"; q2=q_gp, fit_s=t)
            check!(s, "$name: Q2(JLibKriging) - Q2(GaussianProcesses.jl)", q_lk - q_gp > -0.02;
                       value=q_lk - q_gp, expected="> -0.02", severity="warn")
        catch e
            check!(s, "$name: GaussianProcesses.jl fit", false; value=string(typeof(e)), severity="warn")
        end
    end
end
have_gp || check!(s, "GaussianProcesses.jl available", false; value="absent", severity="warn")
foreach(r -> println(join(r[3:6], "  ")), s.metrics)

[WARN] branin: GaussianProcesses.jl fit: value=MethodError expected  


[WARN] hartmann3: GaussianProcesses.jl fit: value=MethodError expected  
[WARN] borehole: GaussianProcesses.jl fit: value=MethodError expected  


branin  JLibKriging  q2  0.999036
branin  JLibKriging  fit_s  0.00236526
branin  JLibKriging BFGS10  q2  0.999035
branin  JLibKriging BFGS10  fit_s  0.00919445
hartmann3  JLibKriging  q2  0.976727
hartmann3  JLibKriging  fit_s  0.00640089
hartmann3  JLibKriging BFGS10  q2  0.976728
hartmann3  JLibKriging BFGS10  fit_s  0.0258171
borehole  JLibKriging  q2  0.999098
borehole  JLibKriging  fit_s  0.0164527
borehole  JLibKriging BFGS10  q2  0.999098
borehole  JLibKriging BFGS10  fit_s  0.13933


## Summary

The metrics of this notebook, the number of checks per status, and every check that
did not pass (FAIL: error-level, fails the run; WARN: warn-level; KNOWN: listed in
`tools/known_issues.csv`, with its issue; SKIP: feature absent from the installed version).
All of them are also written to `results/vs-GaussianProcesses_Kriging_julia.checks.csv` and `.metrics.csv`.

In [4]:
suite_summary(s)

branin                      JLibKriging                 

q2          0.999036
branin                      JLibKriging                 fit_s       0.00236526
branin                      JLibKriging BFGS10          q2          0.999035
branin                      JLibKriging BFGS10          fit_s       0.00919445
hartmann3                   JLibKriging                 q2          0.976727
hartmann3                   JLibKriging                 fit_s       0.00640089
hartmann3                   JLibKriging BFGS10          q2          0.976728
hartmann3                   JLibKriging BFGS10          fit_s       0.0258171
borehole                    JLibKriging                 q2          0.999098
borehole                    JLibKriging                 fit_s       0.0164527
borehole                    JLibKriging BFGS10          q2          0.999098
borehole                    JLibKriging BFGS10          fit_s       0.13933
6 checks: PASS 3, FAIL 0, WARN 3, KNOWN 0, SKIP 0
WARN   warn   branin: GaussianProcesses.jl fit: value MethodError, expected

In [5]:
suite_finalize!(s)

vs-GaussianProcesses_Kriging_julia: 

PASS 3, FAIL 0, WARN 3, KNOWN 0, SKIP 0


true